In [0]:
from pyspark.sql.functions import max, length, col, lit, cast

In [0]:
# Define column limits
column_limits = {
    "tvid": 250,
    "zipcode": 10,
    "dma": 128,
    "tms_episode_id": 256,
    "tivo_episode_id": 256,
    "tms_title": 256,
    "tivo_title": 256,
    "tms_airdate": 40,
    "tivo_airdate": 40,
    "tms_channel_callsign": 30,
    "tivo_channel_callsign": 30,
    "tms_channel_affiliate": 40,
    "tivo_channel_affiliate": 40,
    "is_live": 1,
    "ip_address": 32,
    "reported_input_source": 10,
    "content_type": 10,
    "tuner_content_type": 10,
    "input_category": 16,
    "input_device": 32,
    "app_service": 32,
    "tuner_tms_episode_id": 256,
    "tuner_tivo_episode_id": 256,
    "tuner_tms_title": 256,
    "tuner_tivo_title": 256,
    "tuner_tms_airdate": 40,
    "tuner_tivo_airdate": 40,
    "tuner_tms_channel_callsign": 30,
    "tuner_tivo_channel_callsign": 30,
    "tuner_tms_channel_affiliate": 40,
    "tuner_tivo_channel_affiliate": 40,
    "tuner_is_live": 1,
    "tuner_input_category": 16,
    "tuner_input_device": 32,
    "tuner_app_service": 32,
    "enableaudioacr": 1,
    "vizio_epg_channel_id": 32,
    "vizio_epg_program_id": 32,
    "tms_show_genre": 256,
    "tivo_show_genre": 256,
    "tuner_tms_show_genre": 256,
    "tuner_tivo_show_genre": 256,
    "tms_epi_title": 256,
    "tivo_epi_title": 256,
    # "tuner_tms_epi_title": 256,
    # "tuner_tivo_epi_title": 256,
    "series_id": 256,
    # "tuner_series_id": 256,
}

In [0]:
max_length_exprs = []
for col_name, max_lim in column_limits.items():
    max_length_exprs.append(f"MAX(LENGTH(CAST({col_name} AS STRING))) > {max_lim} AS {col_name}")

In [0]:
max_length_exprs

In [0]:
# Step 1: Find which hours have any column exceeding character limits
# Produces one row per session_start_hour with a 1/0 flag per column

case_exprs = []
for col_name, max_lim in column_limits.items():
    case_exprs.append(
        f"CASE WHEN MAX(LENGTH(CAST(`{col_name}` AS STRING))) > {max_lim} THEN 1 ELSE 0 END AS `{col_name}`"
    )

In [0]:
flag_columns = list(column_limits.keys())
violated_columns_expr = "filter(array(" + ", ".join([f"IF({col_name} = 1, '{col_name}', NULL)" for col_name in flag_columns]) + "), x -> x IS NOT NULL)"
violated_count_expr = " + ".join([f"{col_name}" for col_name in flag_columns])

In [0]:
from datetime import datetime, timedelta

# Iterate one week at a time over the last 2 years
start_date = datetime.now() - timedelta(days=730)
end_date = datetime.now()
current = start_date
week_num = 0

while current < end_date:
    week_end = min(current + timedelta(days=7), end_date)
    week_start_str = current.strftime('%Y-%m-%d')
    week_end_str = week_end.strftime('%Y-%m-%d')

    violations_query = f"""
    WITH hourly_flags AS (
        SELECT session_start_hour,
            {', '.join(case_exprs)}
        FROM prod.detection.viewing_content_golden
        WHERE session_start_hour >= '{week_start_str}'
          AND session_start_hour < '{week_end_str}'
        GROUP BY session_start_hour
    )
    SELECT
        session_start_hour,
        {', '.join([f'{col_name}' for col_name in flag_columns])},
        {violated_columns_expr} AS violated_columns,
        ({violated_count_expr}) AS violated_column_count
    FROM hourly_flags
    WHERE ({violated_count_expr}) > 0
    ORDER BY session_start_hour
    """

    week_df = spark.sql(violations_query)

    # Append to persistent table
    if week_num == 0:
        week_df.write.mode("overwrite").saveAsTable("dev.public.viewing_content_golden_char_violations")
    else:
        week_df.write.mode("append").insertInto("dev.public.viewing_content_golden_char_violations")

    week_num += 1
    if week_num % 10 == 0:
        print(f"  Processed {week_num} weeks through {week_end_str} ({datetime.now().strftime('%H:%M:%S')})")

    current = week_end

print(f"Done! Scanned {week_num} weeks total.")

In [0]:
violations_df = spark.sql(violations_query)
violations_pdf = violations_df.toPandas()

print(f"Hours with at least one violation: {len(violations_pdf)}")
display(violations_df)

In [0]:
# Persist violations table for reference
violations_df.write.mode("overwrite").saveAsTable("dev.public.viewing_content_golden_char_violations")
print("Violations table saved to dev.public.viewing_content_golden_char_violations")

In [0]:
# Step 3: For each violating hour, trim only the columns that exceed limits
# Processes one hour at a time to avoid large transactions

from datetime import datetime

violating_hours = violations_pdf['session_start_hour'].tolist()
print(f"Processing {len(violating_hours)} hours...")

for i, hour in enumerate(violating_hours):
    # Identify which columns are violated for this hour
    row = violations_pdf[violations_pdf['session_start_hour'] == hour].iloc[0]
    violated_cols = [col_name for col_name in column_limits if row[col_name] == 1]

    if not violated_cols:
        continue

    # Build SET clause: only trim violated columns
    set_clauses = [f"`{col_name}` = SUBSTRING(CAST(`{col_name}` AS STRING), 1, {column_limits[col_name]})" for col_name in violated_cols]

    update_sql = f"""
    UPDATE prod.detection.viewing_content_golden
    SET {', '.join(set_clauses)}
    WHERE session_start_hour = '{hour}'
      AND ({' OR '.join([f'LENGTH(CAST(`{col_name}` AS STRING)) > {column_limits[col_name]}' for col_name in violated_cols])})
    """

    spark.sql(update_sql)

    if (i + 1) % 100 == 0 or (i + 1) == len(violating_hours):
        print(f"  Processed {i + 1}/{len(violating_hours)} hours ({datetime.now().strftime('%H:%M:%S')})")

print("Done! All violations fixed.")